In [1]:
import joblib
import pandas as pd

from credit_monitoring import config

model = joblib.load(config.MODEL_PATH)

In [2]:
# Mechanism: what the model learned about employment categories
names = model.named_steps["preprocess"].get_feature_names_out()
coefs = pd.Series(model.named_steps["model"].coef_[0], index=names)
coefs[
    coefs.index.str.contains("NOT_EMPLOYED|Unemployed|Pensioner|Working")
].sort_values()

cat__NAME_INCOME_TYPE_Pensioner      -0.391948
cat__EMPLOYMENT_GROUP_NOT_EMPLOYED   -0.387579
cat__NAME_INCOME_TYPE_Working        -0.167362
cat__NAME_INCOME_TYPE_Unemployed      0.003516
dtype: float64

In [3]:
# Effect: the same clients before (control) and after (data drift) losing their job
control = pd.read_csv(config.PREDICTIONS_CONTROL_PATH)
drifted = pd.read_csv(config.PREDICTIONS_DRIFTED_PATH)
merged = control.merge(drifted, on=config.ID_COLUMN, suffixes=("_control", "_drift"))

newly_unemployed = merged[
    (merged["NAME_INCOME_TYPE_drift"] == "Unemployed")
    & (merged["NAME_INCOME_TYPE_control"] != "Unemployed")
]

pd.DataFrame(
    {
        "control": [
            newly_unemployed["RISK_SCORE_control"].mean(),
            newly_unemployed["APPROVED_control"].mean(),
            newly_unemployed["TARGET_control"].mean(),
        ],
        "data_drift": [
            newly_unemployed["RISK_SCORE_drift"].mean(),
            newly_unemployed["APPROVED_drift"].mean(),
            newly_unemployed["TARGET_drift"].mean(),
        ],
    },
    index=["mean_risk_score", "approval_rate", "default_rate"],
).round(4)

,control,data_drift
mean_risk_score,0.0973,0.0947
approval_rate,0.8567,0.8645
default_rate,0.0975,0.2940
